In [ ]:
import os, sys, torch, platform
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
for pkg in ["transformers", "peft", "bitsandbytes", "accelerate", "datasets", "huggingface_hub"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'installed')}")
    except Exception as e:
        print(f"{pkg}: ERROR {e}")
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130 CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM GB: 39.49
transformers: 5.22.1
peft: 0.21.2
bitsandbytes: ERROR No module named 'bitsandbytes'
accelerate: 1.15.0
datasets: 4.8.5
huggingface_hub: 1.33.0
HF_TOKEN set: False
Drive mounted: False


In [ ]:
!pip install -q bitsandbytes
import os, bitsandbytes
print("bitsandbytes:", bitsandbytes.__version__)
assert os.environ.get("HF_TOKEN"), "HF_TOKEN must be set in environment or Colab Secrets"
from huggingface_hub import HfApi, hf_hub_download
api = HfApi(token=os.environ["HF_TOKEN"])
info = api.model_info("google/gemma-2-2b-it")
print("Gemma-2-2b-it SHA:", info.sha)
assert info.sha == "299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8"
cfg = hf_hub_download("google/gemma-2-2b-it", "config.json", revision=info.sha, token=os.environ["HF_TOKEN"])
print("Verified config.json:", cfg)


bitsandbytes: 0.50.2
Gemma-2-2b-it SHA: 299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

Verified config.json: /root/.cache/huggingface/hub/models--google--gemma-2-2b-it/snapshots/299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8/config.json


In [ ]:
import hashlib, base64, lzma, json, pathlib, shutil
from collections import Counter

b64_data = open("/content/lz2.b64").read()
assert len(b64_data) == 39288, f"Expected 39288, got {len(b64_data)}"
bundle = json.loads(lzma.decompress(base64.b64decode(b64_data)).decode("utf-8"))

repo_root = pathlib.Path("/content/motherlypulse_fresh_repo")
if repo_root.exists():
    shutil.rmtree(repo_root)
repo_root.mkdir(parents=True, exist_ok=True)

for rel_path, content in bundle.items():
    norm_path = rel_path.replace("\\", "/")
    p = repo_root / norm_path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8", newline="\n")

expected_hashes = {
    "data/test_en.jsonl": "276f076f19d84600f840b05bb5e7cf583b7815ee11b39981ac6612950741b739",
    "data/test_pcm.jsonl": "fffdea0494d3f384264bf26527cf0c39d5ac97f8385b48ae61337d2a3720829f",
    "data/test_yo.jsonl": "c69cf9d77dc473148e6e6c4222aa0a4551c374bd93b2756d3bd0ed3f6ab64764",
    "MotherlyPulse_Data/04_Conversation_Data/fixed_15_evaluation_prompts.json": "7830ab3d7b8b0f601a464af34387675a53a279d59688e26592321f5b9529e7bc",
}
for rel_path, exp in expected_hashes.items():
    actual = hashlib.sha256((repo_root / rel_path).read_bytes()).hexdigest()
    assert actual == exp, f"Hash mismatch for {rel_path}: {actual} != {exp}"
    print(f"SHA256 OK: {rel_path} = {actual[:12]}...")

def norm_lang(val):
    v = (val or "").strip().lower()
    if v in ("en", "english"): return "en"
    if v in ("yo", "yoruba", "yorùbá"): return "yo"
    if v in ("pcm", "pidgin", "nigerian pidgin"): return "pcm"
    return v or "unknown"

eval_items = []
for lang, fname in [("en", "data/test_en.jsonl"), ("yo", "data/test_yo.jsonl"), ("pcm", "data/test_pcm.jsonl")]:
    for idx, line in enumerate((repo_root / fname).read_text(encoding="utf-8").splitlines()):
        if not line.strip():
            continue
        obj = json.loads(line)
        eval_items.append({
            "id": obj.get("pair_id", f"test_{lang}_{idx+1}"),
            "source": fname,
            "lang": lang,
            "triage": obj["triage_level"],
            "prompt": obj["prompt"],
            "expected": obj.get("target", ""),
        })

fixed_15 = json.loads((repo_root / "MotherlyPulse_Data/04_Conversation_Data/fixed_15_evaluation_prompts.json").read_text(encoding="utf-8"))
for item in fixed_15:
    eval_items.append({
        "id": item["id"],
        "source": "fixed_15_evaluation_prompts.json",
        "lang": norm_lang(item.get("lang")),
        "triage": item.get("expected_triage", item.get("triage", "unknown")),
        "prompt": item["prompt"],
        "expected": "",
    })

print(f"Total eval_items: {len(eval_items)}")
print("By lang:", dict(Counter(x["lang"] for x in eval_items)))
print("By triage:", dict(Counter(x["triage"] for x in eval_items)))
print("By source:", dict(Counter(x["source"] for x in eval_items)))


SHA256 OK: data/test_en.jsonl = 276f076f19d8...
SHA256 OK: data/test_pcm.jsonl = fffdea0494d3...
SHA256 OK: data/test_yo.jsonl = c69cf9d77dc4...
SHA256 OK: MotherlyPulse_Data/04_Conversation_Data/fixed_15_evaluation_prompts.json = 7830ab3d7b8b...
Total eval_items: 135
By lang: {'en': 45, 'yo': 45, 'pcm': 45}
By triage: {'Emergency': 42, 'Routine': 66, 'Priority': 27}
By source: {'data/test_en.jsonl': 40, 'data/test_yo.jsonl': 40, 'data/test_pcm.jsonl': 40, 'fixed_15_evaluation_prompts.json': 15}


In [ ]:
import gc, hashlib, json, os, torch

# Clean up Phase 2 base_model before Phase 3 QLoRA training
if "base_model" in globals():
    del base_model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

REPO_DIR = "/content/motherlypulse_fresh_repo"
train_files = ["data/train_en.jsonl", "data/train_pcm.jsonl", "data/train_yo.jsonl"]
val_files = ["data/val_en.jsonl", "data/val_pcm.jsonl", "data/val_yo.jsonl"]

train_rows, val_rows = [], []
for rel in train_files:
    p = os.path.join(REPO_DIR, rel)
    with open(p, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    train_rows.extend(rows)
    print(f"Verified {rel}: {len(rows)} rows, sha256={hashlib.sha256(open(p,'rb').read()).hexdigest()[:12]}")

for rel in val_files:
    p = os.path.join(REPO_DIR, rel)
    with open(p, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    val_rows.extend(rows)
    print(f"Verified {rel}: {len(rows)} rows, sha256={hashlib.sha256(open(p,'rb').read()).hexdigest()[:12]}")

print(f"Total train_rows: {len(train_rows)}, Total val_rows: {len(val_rows)}")
print("Sample train keys:", list(train_rows[0].keys()))
print(f"GPU allocated after cleanup: {torch.cuda.memory_allocated()/1e9:.3f} GB")

Verified data/train_en.jsonl: 140 rows, sha256=d40d8ad441f4
Verified data/train_pcm.jsonl: 140 rows, sha256=58cf1a5689f4
Verified data/train_yo.jsonl: 140 rows, sha256=961e0c8541a2
Verified data/val_en.jsonl: 20 rows, sha256=c5666c21a1bc
Verified data/val_pcm.jsonl: 20 rows, sha256=5c141e499e4f
Verified data/val_yo.jsonl: 20 rows, sha256=9ca0df0cca58
Total train_rows: 420, Total val_rows: 60
Sample train keys: ['pair_id', 'user_group', 'language', 'triage_level', 'passage_id', 'split', 'review_status', 'prompt', 'target', 'user_message', 'reference_passage']
GPU allocated after cleanup: 0.009 GB


In [ ]:
import base64, hashlib, json, lzma, os, time, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-2-2b-it"
MODEL_REV = "299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8"
OUT_DIR = "/content/fresh_chewie_1_2"
CKPT_DIR = f"{OUT_DIR}/checkpoints"
ADAPTER_DIR = f"{OUT_DIR}/chewie_adapter"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_chat_example(row, max_length=1024):
    prompt_msgs = [{"role": "user", "content": row["prompt"]}]
    full_msgs = [
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["target"]},
    ]
    prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_msgs, tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=max_length)["input_ids"]
    prompt_len = min(len(prompt_ids), len(full_ids))
    labels = [-100] * prompt_len + full_ids[prompt_len:]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

train_ds = Dataset.from_list([tokenize_chat_example(r) for r in train_rows]).shuffle(seed=42)
val_ds = Dataset.from_list([tokenize_chat_example(r) for r in val_rows])
print(f"Tokenized train_ds={len(train_ds)}, val_ds={len(val_ds)}")
print("Sample non-masked label count:", sum(1 for x in train_ds[0]["labels"] if x != -100))

def collate_fn(features):
    max_len = max(len(f["input_ids"]) for f in features)
    pad_id = tokenizer.pad_token_id
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for f in features:
        pad_n = max_len - len(f["input_ids"])
        batch["input_ids"].append(f["input_ids"] + [pad_id] * pad_n)
        batch["attention_mask"].append(f["attention_mask"] + [0] * pad_n)
        batch["labels"].append(f["labels"] + [-100] * pad_n)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
train_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REV,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ.get("HF_TOKEN"),
)
train_model.config.use_cache = False
train_model = prepare_model_for_kbit_training(train_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
peft_model = get_peft_model(train_model, lora_config)
peft_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    max_steps=80,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=40,
    report_to="none",
    seed=42,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_result = trainer.train()
train_elapsed_s = round(time.time() - t0, 2)
peak_vram_gb = round(torch.cuda.max_memory_allocated() / 1e9, 3)

peft_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

adapter_hashes = {}
for fn in sorted(os.listdir(ADAPTER_DIR)):
    fp = os.path.join(ADAPTER_DIR, fn)
    if os.path.isfile(fp):
        adapter_hashes[fn] = {
            "bytes": os.path.getsize(fp),
            "sha256": hashlib.sha256(open(fp, "rb").read()).hexdigest(),
        }

log_payload = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REV,
    "train_rows": len(train_rows),
    "val_rows": len(val_rows),
    "train_elapsed_s": train_elapsed_s,
    "peak_vram_gb": peak_vram_gb,
    "train_metrics": train_result.metrics,
    "log_history": trainer.state.log_history,
    "adapter_files": adapter_hashes,
}
with open(f"{OUT_DIR}/training_log.json", "w", encoding="utf-8") as f:
    json.dump(log_payload, f, indent=2)

print(f"Training completed in {train_elapsed_s}s, peak VRAM={peak_vram_gb} GB")
print("Log history:", json.dumps(trainer.state.log_history, indent=2))
print("Adapter files:", json.dumps(adapter_hashes, indent=2))
log_b64 = base64.b64encode(lzma.compress(json.dumps(log_payload, indent=2).encode("utf-8"))).decode("ascii")
print("TRAINING_LOG_LZMA_B64_START")
print(log_b64)
print("TRAINING_LOG_LZMA_B64_END")

Tokenized train_ds=420, val_ds=60
Sample non-masked label count: 54


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
20,2.485635,2.205514
40,2.347307,2.084949
60,2.156967,2.038267
80,2.245356,2.015061


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training completed in 148.34s, peak VRAM=11.098 GB
Log history: [
  {
    "loss": 3.1088661193847655,
    "grad_norm": 1.6842844486236572,
    "learning_rate": 0.0001775,
    "epoch": 0.19047619047619047,
    "step": 10
  },
  {
    "loss": 2.485634994506836,
    "grad_norm": 1.9763375520706177,
    "learning_rate": 0.0001525,
    "epoch": 0.38095238095238093,
    "step": 20
  },
  {
    "eval_loss": 2.205514430999756,
    "eval_runtime": 1.2718,
    "eval_samples_per_second": 47.179,
    "eval_steps_per_second": 6.29,
    "epoch": 0.38095238095238093,
    "step": 20
  },
  {
    "loss": 2.617483139038086,
    "grad_norm": 1.620717167854309,
    "learning_rate": 0.0001275,
    "epoch": 0.5714285714285714,
    "step": 30
  },
  {
    "loss": 2.3473072052001953,
    "grad_norm": 2.0750668048858643,
    "learning_rate": 0.0001025,
    "epoch": 0.7619047619047619,
    "step": 40
  },
  {
    "eval_loss": 2.084949016571045,
    "eval_runtime": 1.2511,
    "eval_samples_per_second": 47.959,
